In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ==========================================
# LOAD FINALIZED MODULE RESULTS
# ==========================================

financial = pd.read_csv(
    "mplads_anomaly_results_2026_27.csv"
)

payment = pd.read_csv(
    "payment_anomaly_results_2026_27.csv"
)

execution = pd.read_csv(
    "execution_anomaly_results_2026_27.csv"
)

print("Financial shape :", financial.shape)
print("Payment shape   :", payment.shape)
print("Execution shape :", execution.shape)

Financial shape : (20016, 16)
Payment shape   : (8631, 18)
Execution shape : (20016, 31)


In [3]:
print("========== FINANCIAL ==========")
print(financial.columns.tolist())

print("\n========== PAYMENT ==========")
print(payment.columns.tolist())

print("\n========== EXECUTION ==========")
print(execution.columns.tolist())

========== FINANCIAL ==========
['work_id', 'financial_year', 'sanction_amount', 'total_disbursed_amount', 'payment_count', 'in_progress_payment_count', 'in_progress_payment_amount', 'payment_to_sanction_ratio', 'payment_minus_sanction_amount', 'average_payment_amount', 'maximum_payment_amount', 'minimum_payment_amount', 'payment_amount_std', 'payment_duration_days', 'anomaly_score', 'baseline_anomaly']

========== PAYMENT ==========
['payment_id', 'work_id', 'vendor_name_normalized', 'financial_year', 'expenditure_date', 'fund_disbursed_amount', 'sanction_amount', 'payment_sequence_number', 'days_since_previous_payment', 'days_since_sanction', 'prior_disbursed_amount', 'prior_vendor_count', 'payment_share_of_prior_disbursed_amount', 'vendor_previous_payment_count', 'vendor_previous_work_count', 'vendor_previous_total_amount', 'anomaly_score', 'payment_anomaly']

========== EXECUTION ==========
['work_id', 'house', 'mp_key', 'state', 'ida', 'work_category', 'work_title', 'financial_yea

In [4]:
# ==========================================
# WORK ID CONSISTENCY CHECK
# ==========================================

financial_ids = set(financial["work_id"])
payment_ids = set(payment["work_id"])
execution_ids = set(execution["work_id"])

print("Financial works :", len(financial_ids))
print("Payment works   :", len(payment_ids))
print("Execution works :", len(execution_ids))

print("\nFinancial ∩ Payment   :",
      len(financial_ids & payment_ids))

print("Financial ∩ Execution :",
      len(financial_ids & execution_ids))

print("Payment ∩ Execution   :",
      len(payment_ids & execution_ids))

Financial works : 20016
Payment works   : 6972
Execution works : 20016

Financial ∩ Payment   : 6972
Financial ∩ Execution : 20016
Payment ∩ Execution   : 6972


In [5]:
# ==========================================
# COMBINE THE THREE MODULES
# ==========================================

risk_df = financial[
    [
        "work_id",
        "anomaly_score"
    ]
].copy()

risk_df = risk_df.merge(
    payment[
        [
            "work_id",
            "payment_anomaly_score"
        ]
    ],
    on="work_id",
    how="inner"
)

risk_df = risk_df.merge(
    execution[
        [
            "work_id",
            "execution_anomaly_score",
            "execution_anomaly_flag",
            "consistency_review_flag",
            "payment_exceeds_sanction_flag",
            "execution_anomaly_reason"
        ]
    ],
    on="work_id",
    how="inner"
)

print("Combined dataset:", risk_df.shape)

print("\nMissing values:")
print(risk_df.isna().sum())

KeyError: "['payment_anomaly_score'] not in index"

In [6]:
print("PAYMENT COLUMNS:")
print(payment.columns.tolist())

PAYMENT COLUMNS:
['payment_id', 'work_id', 'vendor_name_normalized', 'financial_year', 'expenditure_date', 'fund_disbursed_amount', 'sanction_amount', 'payment_sequence_number', 'days_since_previous_payment', 'days_since_sanction', 'prior_disbursed_amount', 'prior_vendor_count', 'payment_share_of_prior_disbursed_amount', 'vendor_previous_payment_count', 'vendor_previous_work_count', 'vendor_previous_total_amount', 'anomaly_score', 'payment_anomaly']


In [7]:
print("\nFIRST 5 PAYMENT ROWS:")
display(payment.head())


FIRST 5 PAYMENT ROWS:


,payment_id,work_id,vendor_name_normalized,financial_year,expenditure_date,fund_disbursed_amount,sanction_amount,payment_sequence_number,days_since_previous_payment,days_since_sanction,prior_disbursed_amount,prior_vendor_count,payment_share_of_prior_disbursed_amount,vendor_previous_payment_count,vendor_previous_work_count,vendor_previous_total_amount,anomaly_score,payment_anomaly
0,PAY_LS_000126,WS/MP477/2026-2027/274288,BISHT TRADING COMPANY,2026-2027,2026-09-03,122244.0,200000.0,1,NaN,111,0.0,0,NaN,1,1,4.901100e+04,0.465310,False
1,PAY_LS_000128,WS/MP18095/2026-2027/246711,TATA MOTORS,2026-2027,2026-09-03,1601555.0,1841700.0,1,NaN,3,0.0,0,NaN,82,74,1.247678e+08,0.624785,True
2,PAY_LS_000129,WS/MP18272/2026-2027/222318,RAJAT SUBHRA MANDAL,2026-2027,2026-08-17,4881631.0,4882225.0,1,NaN,20,0.0,0,NaN,1,1,1.500000e+06,0.579660,False
3,PAY_LS_000130,WS/MP18072/2026-2027/261891,BIRENDRA KUMAR MAHATO,2026-2027,2026-08-31,375000.0,750000.0,1,NaN,48,0.0,0,NaN,202,119,3.857250e+07,0.536625,False
4,PAY_LS_000131,WS/MP18072/2026-2027/277323,BIRENDRA KUMAR MAHATO,2026-2027,2026-08-31,190000.0,380000.0,1,NaN,33,0.0,0,NaN,203,120,3.894750e+07,0.530783,False


In [8]:
# ==========================================
# PREPARE PAYMENT RISK AT WORK LEVEL
# ==========================================

print("Payment rows:", len(payment))
print("Unique works:", payment["work_id"].nunique())

# A work may have multiple payments.
# For combined risk, use the highest payment anomaly
# score observed for each work.

payment_work = (
    payment
    .groupby("work_id")
    .agg(
        payment_anomaly_score=("anomaly_score", "max"),
        payment_anomaly_flag=("payment_anomaly", "max"),
        payment_count=("payment_id", "count")
    )
    .reset_index()
)

print("\nPayment work-level dataset:")
print(payment_work.shape)

print("\nPayment anomaly score:")
print(
    payment_work["payment_anomaly_score"].describe()
)

print("\nPayment anomaly flags:")
print(
    payment_work["payment_anomaly_flag"].value_counts()
)

Payment rows: 8631
Unique works: 6972

Payment work-level dataset:
(6972, 4)

Payment anomaly score:
count    6972.000000
mean        0.382202
std         0.068662
min         0.321363
25%         0.331261
50%         0.347687
75%         0.422816
max         0.759301
Name: payment_anomaly_score, dtype: float64

Payment anomaly flags:
payment_anomaly_flag
False    6934
True       38
Name: count, dtype: int64


In [9]:
# ==========================================
# COMBINE THREE MODULES AT WORK LEVEL
# ==========================================

risk_df = financial[
    [
        "work_id",
        "anomaly_score"
    ]
].copy()

risk_df = risk_df.merge(
    payment_work[
        [
            "work_id",
            "payment_anomaly_score",
            "payment_anomaly_flag",
            "payment_count"
        ]
    ],
    on="work_id",
    how="left"
)

risk_df = risk_df.merge(
    execution[
        [
            "work_id",
            "execution_anomaly_score",
            "execution_anomaly_flag",
            "consistency_review_flag",
            "payment_exceeds_sanction_flag",
            "execution_anomaly_reason"
        ]
    ],
    on="work_id",
    how="left"
)

print("Combined dataset:", risk_df.shape)

print("\nMissing values:")
print(risk_df.isna().sum())

Combined dataset: (20016, 10)

Missing values:
work_id                              0
anomaly_score                        0
payment_anomaly_score            13044
payment_anomaly_flag             13044
payment_count                    13044
execution_anomaly_score              0
execution_anomaly_flag               0
consistency_review_flag              0
payment_exceeds_sanction_flag        0
execution_anomaly_reason             0
dtype: int64


In [10]:
# ==========================================
# HANDLE PAYMENT DATA AVAILABILITY
# ==========================================

# Record whether payment information exists
risk_df["payment_data_available"] = (
    risk_df["payment_count"].fillna(0) > 0
).astype(int)

# Fill payment anomaly values only for works
# where there is no payment record.
risk_df["payment_anomaly_score"] = (
    risk_df["payment_anomaly_score"].fillna(0)
)

risk_df["payment_anomaly_flag"] = (
    risk_df["payment_anomaly_flag"].fillna(False)
    .astype(int)
)

risk_df["payment_count"] = (
    risk_df["payment_count"].fillna(0)
    .astype(int)
)

print("PAYMENT DATA AVAILABILITY")
print("=" * 50)

print(
    risk_df["payment_data_available"]
    .value_counts()
    .rename({
        0: "No payment data",
        1: "Payment data available"
    })
)

print("\nMissing values:")
print(
    risk_df[
        [
            "payment_anomaly_score",
            "payment_anomaly_flag",
            "payment_count"
        ]
    ].isna().sum()
)

PAYMENT DATA AVAILABILITY
payment_data_available
No payment data           13044
Payment data available     6972
Name: count, dtype: int64

Missing values:
payment_anomaly_score    0
payment_anomaly_flag     0
payment_count            0
dtype: int64


In [11]:
# ==========================================
# PAYMENT WORK-LEVEL SUMMARY
# ==========================================

print("Payment-anomalous works:",
      risk_df["payment_anomaly_flag"].sum())

print("Works with payment data:",
      risk_df["payment_data_available"].sum())

print("Works without payment data:",
      (risk_df["payment_data_available"] == 0).sum())

print("\nPayment anomaly score distribution:")
print(
    risk_df[
        risk_df["payment_data_available"] == 1
    ]["payment_anomaly_score"].describe()
)

Payment-anomalous works: 38
Works with payment data: 6972
Works without payment data: 13044

Payment anomaly score distribution:
count    6972.000000
mean        0.382202
std         0.068662
min         0.321363
25%         0.331261
50%         0.347687
75%         0.422816
max         0.759301
Name: payment_anomaly_score, dtype: float64


In [12]:
# ==========================================
# NORMALIZE FINANCIAL + PAYMENT RISK
# ==========================================

# Financial risk:
# All 20,016 works have a financial anomaly score.
risk_df["financial_risk_0_100"] = (
    risk_df["anomaly_score"]
    .rank(pct=True)
    * 100
)

# Payment risk:
# Only works that actually have payment data
# should participate in the payment-score ranking.
payment_mask = risk_df["payment_data_available"] == 1

risk_df["payment_risk_0_100"] = 0.0

risk_df.loc[payment_mask, "payment_risk_0_100"] = (
    risk_df.loc[payment_mask, "payment_anomaly_score"]
    .rank(pct=True)
    * 100
)

print("FINANCIAL RISK")
print(
    risk_df["financial_risk_0_100"].describe()
)

print("\nPAYMENT RISK — works with payment data")
print(
    risk_df.loc[
        payment_mask,
        "payment_risk_0_100"
    ].describe()
)

print("\nWorks without payment data:")
print(
    (risk_df["payment_data_available"] == 0).sum()
)

FINANCIAL RISK
count    20016.000000
mean        50.002498
std         28.852662
min          0.172362
25%         25.044964
50%         45.498601
75%         74.990008
max        100.000000
Name: financial_risk_0_100, dtype: float64

PAYMENT RISK — works with payment data
count    6972.000000
mean       50.007172
std        28.869580
min         0.014343
25%        25.010757
50%        50.050201
75%        75.003586
max       100.000000
Name: payment_risk_0_100, dtype: float64

Works without payment data:
13044


In [13]:
# ==========================================
# EXECUTION RISK 0–100
# ==========================================

risk_df["execution_risk_0_100"] = np.where(
    risk_df["execution_anomaly_flag"] == 1,
    100,
    0
)

print("EXECUTION RISK DISTRIBUTION")
print(
    risk_df["execution_risk_0_100"].value_counts()
)

EXECUTION RISK DISTRIBUTION
execution_risk_0_100
0    20016
Name: count, dtype: int64


In [15]:
# ==========================================
# REVISED COMBINED RISK SCORE
# ==========================================

# Start with Financial Risk
risk_df["combined_base_risk"] = (
    risk_df["financial_risk_0_100"]
)

# For works that have payment data,
# combine Financial + Payment equally.
payment_available = (
    risk_df["payment_data_available"] == 1
)

risk_df.loc[payment_available, "combined_base_risk"] = (
    0.50 * risk_df.loc[
        payment_available,
        "financial_risk_0_100"
    ]
    +
    0.50 * risk_df.loc[
        payment_available,
        "payment_risk_0_100"
    ]
)

print("REVISED COMBINED BASE RISK")
print("=" * 55)

print(
    risk_df["combined_base_risk"].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

REVISED COMBINED BASE RISK
count    20016.000000
mean        51.707633
std         24.829550
min          0.251127
50%         48.802596
75%         73.106515
90%         87.856720
95%         92.238709
99%         97.552849
max         99.997502
Name: combined_base_risk, dtype: float64


In [18]:
# ==========================================
# SET FINAL RISK SCORE
# ==========================================

risk_df["final_risk_score"] = risk_df["combined_base_risk"]

# Payment exceeds sanction = CRITICAL override
compliance_mask = (
    risk_df["payment_exceeds_sanction_flag"] == 1
)

risk_df.loc[
    compliance_mask,
    "final_risk_score"
] = 100

print("Final risk score created successfully.")

print(
    risk_df["final_risk_score"].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print(
    "\nPayment > sanction overrides:",
    compliance_mask.sum()
)

Final risk score created successfully.
count    20016.000000
mean        51.707633
std         24.829550
min          0.251127
50%         48.802596
75%         73.106515
90%         87.856720
95%         92.238709
99%         97.552849
max         99.997502
Name: final_risk_score, dtype: float64

Payment > sanction overrides: 0


In [19]:
# ==========================================
# FINAL RISK THRESHOLDS
# ==========================================

P75 = risk_df["final_risk_score"].quantile(0.75)
P90 = risk_df["final_risk_score"].quantile(0.90)
P95 = risk_df["final_risk_score"].quantile(0.95)

print("RISK THRESHOLDS")
print("=" * 50)

print(f"P75  = {P75:.6f}")
print(f"P90  = {P90:.6f}")
print(f"P95  = {P95:.6f}")

RISK THRESHOLDS
P75  = 73.106515
P90  = 87.856720
P95  = 92.238709


In [20]:
# ==========================================
# RISK LEVEL CLASSIFICATION
# ==========================================

risk_df["risk_level"] = np.select(
    [
        risk_df["final_risk_score"] >= P95,
        risk_df["final_risk_score"] >= P90,
        risk_df["final_risk_score"] >= P75
    ],
    [
        "VERY_HIGH",
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

# Compliance override
risk_df.loc[
    compliance_mask,
    "risk_level"
] = "CRITICAL"

print("\nRISK LEVEL DISTRIBUTION")
print("=" * 50)

print(
    risk_df["risk_level"].value_counts()
)


RISK LEVEL DISTRIBUTION
risk_level
LOW          14989
MEDIUM        3025
VERY_HIGH     1003
HIGH           999
Name: count, dtype: int64


In [21]:
# ==========================================
# TOP 25 HIGHEST-RISK WORKS
# ==========================================

top_risk = (
    risk_df
    .sort_values(
        "final_risk_score",
        ascending=False
    )
    .head(25)
)

display(
    top_risk[
        [
            "work_id",
            "financial_risk_0_100",
            "payment_risk_0_100",
            "execution_risk_0_100",
            "combined_base_risk",
            "final_risk_score",
            "risk_level",
            "payment_data_available",
            "payment_anomaly_flag",
            "consistency_review_flag",
            "payment_exceeds_sanction_flag",
            "execution_anomaly_reason"
        ]
    ]
)

,work_id,financial_risk_0_100,payment_risk_0_100,execution_risk_0_100,combined_base_risk,final_risk_score,risk_level,payment_data_available,payment_anomaly_flag,consistency_review_flag,payment_exceeds_sanction_flag,execution_anomaly_reason
1,WS/MP507/2026-2027/290535,99.995004,100.000000,0,99.997502,99.997502,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...
20,WS/MP203/2026-2027/284964,99.900080,99.971314,0,99.935697,99.935697,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...
4,WS/MP18368/2026-2027/296403,99.980016,99.885255,0,99.932636,99.932636,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...
18,WS/MP492/2026-2027/282269,99.910072,99.942628,0,99.926350,99.926350,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...
23,WS/MP190/2026-2027/282964,99.885092,99.956971,0,99.921031,99.921031,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...
32,WS/MP187/2026-2027/283439,99.840128,99.985657,0,99.912892,99.912892,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...
15,WS/MP163/2026-2027/70361,99.925060,99.827883,0,99.876471,99.876471,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...
40,WS/MP195/2026-2027/299885,99.800160,99.899598,0,99.849879,99.849879,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...
5,WS/MP492/2026-2027/294027,99.975020,99.713138,0,99.844079,99.844079,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...
12,WS/MP18210/2026-2027/295272,99.940048,99.698795,0,99.819422,99.819422,VERY_HIGH,1,1,0,0,No execution anomaly or consistency issue dete...


In [22]:
# ==========================================
# TOP-RISK CASE VALIDATION
# ==========================================

top_ids = top_risk["work_id"].tolist()

# Financial details
financial_details = financial[
    financial["work_id"].isin(top_ids)
].copy()

# Payment details for those works
payment_details = payment[
    payment["work_id"].isin(top_ids)
].copy()

print("TOP-RISK FINANCIAL DETAILS")
display(financial_details.head(25))

print("\nTOP-RISK PAYMENT DETAILS")
display(payment_details.head(50))

TOP-RISK FINANCIAL DETAILS


,work_id,financial_year,sanction_amount,total_disbursed_amount,payment_count,in_progress_payment_count,in_progress_payment_amount,payment_to_sanction_ratio,payment_minus_sanction_amount,average_payment_amount,maximum_payment_amount,minimum_payment_amount,payment_amount_std,payment_duration_days,anomaly_score,baseline_anomaly
0,WS/MP18435/2026-2027/255473,2026-2027,15000000.0,8112106.0,3,2,17588.0,0.540807,-6887894.0,2.704035e+06,8094518.0,168.0,4.668303e+06,12.0,0.813908,1
1,WS/MP507/2026-2027/290535,2026-2027,49740000.0,35321457.0,8,0,0.0,0.710122,-14418543.0,4.415182e+06,5651433.0,3532146.0,6.865257e+05,0.0,0.803459,1
2,WS/MP188/2026-2027/179557,2026-2027,19933000.0,2710610.0,1,1,2710610.0,0.135986,-17222390.0,2.710610e+06,2710610.0,2710610.0,0.000000e+00,0.0,0.802660,1
3,WS/MP888/2026-2027/251789,2026-2027,9850044.0,9710913.0,1,1,9710913.0,0.985875,-139131.0,9.710913e+06,9710913.0,9710913.0,0.000000e+00,0.0,0.798889,1
4,WS/MP18368/2026-2027/296403,2026-2027,23047200.0,13780800.0,4,0,0.0,0.597938,-9266400.0,3.445200e+06,4276800.0,2376000.0,8.115585e+05,0.0,0.791641,1
5,WS/MP492/2026-2027/294027,2026-2027,14533200.0,8580000.0,1,0,0.0,0.590372,-5953200.0,8.580000e+06,8580000.0,8580000.0,0.000000e+00,0.0,0.790830,1
10,WS/MP249/2026-2027/282367,2026-2027,8737551.0,8737383.0,1,1,8737383.0,0.999981,-168.0,8.737383e+06,8737383.0,8737383.0,0.000000e+00,0.0,0.785231,1
11,WS/MP249/2026-2027/282366,2026-2027,7247584.0,7247383.0,1,1,7247383.0,0.999972,-201.0,7.247383e+06,7247383.0,7247383.0,0.000000e+00,0.0,0.779724,1
12,WS/MP18210/2026-2027/295272,2026-2027,14493600.0,8553600.0,3,0,0.0,0.590164,-5940000.0,2.851200e+06,3326400.0,2376000.0,4.752000e+05,0.0,0.777683,1
15,WS/MP163/2026-2027/70361,2026-2027,14600000.0,13824453.0,1,0,0.0,0.946880,-775547.0,1.382445e+07,13824453.0,13824453.0,0.000000e+00,0.0,0.771845,1



TOP-RISK PAYMENT DETAILS


,payment_id,work_id,vendor_name_normalized,financial_year,expenditure_date,fund_disbursed_amount,sanction_amount,payment_sequence_number,days_since_previous_payment,days_since_sanction,prior_disbursed_amount,prior_vendor_count,payment_share_of_prior_disbursed_amount,vendor_previous_payment_count,vendor_previous_work_count,vendor_previous_total_amount,anomaly_score,payment_anomaly
1,PAY_LS_000128,WS/MP18095/2026-2027/246711,TATA MOTORS,2026-2027,2026-09-03,1601555.0,1841700.0,1,NaN,3,0.0,0,NaN,82,74,1.247678e+08,0.624785,True
2,PAY_LS_000129,WS/MP18272/2026-2027/222318,RAJAT SUBHRA MANDAL,2026-2027,2026-08-17,4881631.0,4882225.0,1,NaN,20,0.0,0,NaN,1,1,1.500000e+06,0.579660,False
195,PAY_LS_000840,WS/MP249/2026-2027/282366,DUTTA ENTERPRISE,2026-2027,2026-09-03,7247383.0,7247584.0,1,NaN,7,0.0,0,NaN,16,16,2.450263e+07,0.612951,False
196,PAY_LS_000841,WS/MP249/2026-2027/282367,DUTTA ENTERPRISE,2026-2027,2026-09-03,8737383.0,8737551.0,1,NaN,7,0.0,0,NaN,17,17,3.175002e+07,0.622121,True
744,PAY_LS_002587,WS/MP18095/2026-2027/246711,ST ANTONYS HS KOKKOTHAMANGALAM,2026-2027,2026-09-03,74587.0,1841700.0,2,0.0,3,1601555.0,1,0.046572,0,0,0.000000e+00,0.527144,False
3565,PAY_LS_018020,WS/MP18198/2026-2027/268853,SHIVAM CONSTRUCTION SURES CHANDRA YADAV,2026-2027,2026-06-30,1009729.0,7759136.0,1,NaN,90,0.0,0,NaN,8,2,9.526378e+06,0.412876,False
3566,PAY_LS_018021,WS/MP18198/2026-2027/268853,SHIVAM CONSTRUCTION SURES CHANDRA YADAV,2026-2027,2026-06-30,1610616.0,7759136.0,2,0.0,90,1009729.0,1,1.595097,9,3,1.053611e+07,0.493092,False
3567,PAY_LS_018022,WS/MP18198/2026-2027/268853,SHIVAM CONSTRUCTION SURES CHANDRA YADAV,2026-2027,2026-07-27,3461928.0,7759136.0,3,27.0,117,2620345.0,1,1.321173,10,3,1.214672e+07,0.579478,False
3568,PAY_LS_018023,WS/MP18198/2026-2027/268853,SHIVAM CONSTRUCTION SURES CHANDRA YADAV,2026-2027,2026-07-27,1303832.0,7759136.0,4,0.0,117,6082273.0,1,0.214366,11,3,1.560865e+07,0.535068,False
5859,PAY_LS_072564,WS/MP492/2026-2027/294027,RAMJI CONSTRUCTION,2026-2027,2026-08-26,8580000.0,14533200.0,1,NaN,12,0.0,0,NaN,57,9,1.597594e+08,0.639075,True


In [23]:
# ==========================================
# TOP-RISK COMPONENT ANALYSIS
# ==========================================

top25 = (
    risk_df
    .sort_values("final_risk_score", ascending=False)
    .head(25)
    .copy()
)

top25["dominant_signal"] = np.select(
    [
        (
            (top25["financial_risk_0_100"] >= top25["payment_risk_0_100"]) &
            (top25["financial_risk_0_100"] >= top25["execution_risk_0_100"])
        ),
        (
            (top25["payment_risk_0_100"] >= top25["financial_risk_0_100"]) &
            (top25["payment_risk_0_100"] >= top25["execution_risk_0_100"])
        )
    ],
    [
        "FINANCIAL",
        "PAYMENT"
    ],
    default="EXECUTION"
)

display(
    top25[
        [
            "work_id",
            "financial_risk_0_100",
            "payment_risk_0_100",
            "execution_risk_0_100",
            "combined_base_risk",
            "final_risk_score",
            "risk_level",
            "dominant_signal",
            "payment_data_available",
            "payment_anomaly_flag",
            "consistency_review_flag",
            "payment_exceeds_sanction_flag"
        ]
    ]
)

,work_id,financial_risk_0_100,payment_risk_0_100,execution_risk_0_100,combined_base_risk,final_risk_score,risk_level,dominant_signal,payment_data_available,payment_anomaly_flag,consistency_review_flag,payment_exceeds_sanction_flag
1,WS/MP507/2026-2027/290535,99.995004,100.000000,0,99.997502,99.997502,VERY_HIGH,PAYMENT,1,1,0,0
20,WS/MP203/2026-2027/284964,99.900080,99.971314,0,99.935697,99.935697,VERY_HIGH,PAYMENT,1,1,0,0
4,WS/MP18368/2026-2027/296403,99.980016,99.885255,0,99.932636,99.932636,VERY_HIGH,FINANCIAL,1,1,0,0
18,WS/MP492/2026-2027/282269,99.910072,99.942628,0,99.926350,99.926350,VERY_HIGH,PAYMENT,1,1,0,0
23,WS/MP190/2026-2027/282964,99.885092,99.956971,0,99.921031,99.921031,VERY_HIGH,PAYMENT,1,1,0,0
32,WS/MP187/2026-2027/283439,99.840128,99.985657,0,99.912892,99.912892,VERY_HIGH,PAYMENT,1,1,0,0
15,WS/MP163/2026-2027/70361,99.925060,99.827883,0,99.876471,99.876471,VERY_HIGH,FINANCIAL,1,1,0,0
40,WS/MP195/2026-2027/299885,99.800160,99.899598,0,99.849879,99.849879,VERY_HIGH,PAYMENT,1,1,0,0
5,WS/MP492/2026-2027/294027,99.975020,99.713138,0,99.844079,99.844079,VERY_HIGH,FINANCIAL,1,1,0,0
12,WS/MP18210/2026-2027/295272,99.940048,99.698795,0,99.819422,99.819422,VERY_HIGH,FINANCIAL,1,1,0,0


In [25]:
# ==========================================
# INVESTIGATION PRIORITY
# ==========================================

risk_df["investigation_priority"] = np.select(
    [
        risk_df["risk_level"] == "CRITICAL",
        risk_df["risk_level"] == "VERY_HIGH",
        risk_df["risk_level"] == "HIGH",
        risk_df["consistency_review_flag"] == 1
    ],
    [
        "CRITICAL_REVIEW",
        "VERY_HIGH_REVIEW",
        "HIGH_REVIEW",
        "CONSISTENCY_REVIEW"
    ],
    default="NORMAL"
)

print("INVESTIGATION PRIORITY")
print("=" * 50)

print(
    risk_df["investigation_priority"].value_counts()
)

INVESTIGATION PRIORITY
investigation_priority
NORMAL                18003
VERY_HIGH_REVIEW       1003
HIGH_REVIEW             999
CONSISTENCY_REVIEW       11
Name: count, dtype: int64


In [26]:
# ==========================================
# FINAL COMBINED RISK ENGINE SUMMARY
# ==========================================

print("=" * 65)
print("MPLADS COMBINED RISK ENGINE — FY 2026-27")
print("=" * 65)

print("\nTotal works:", len(risk_df))

print("\nRisk level distribution:")
print(risk_df["risk_level"].value_counts())

print("\nInvestigation priority:")
print(risk_df["investigation_priority"].value_counts())

print("\nPayment anomaly works:",
      risk_df["payment_anomaly_flag"].sum())

print("Consistency review cases:",
      risk_df["consistency_review_flag"].sum())

print("Payment exceeds sanction:",
      risk_df["payment_exceeds_sanction_flag"].sum())

print("\nFinal score statistics:")
print(
    risk_df["final_risk_score"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

MPLADS COMBINED RISK ENGINE — FY 2026-27

Total works: 20016

Risk level distribution:
risk_level
LOW          14989
MEDIUM        3025
VERY_HIGH     1003
HIGH           999
Name: count, dtype: int64

Investigation priority:
investigation_priority
NORMAL                18003
VERY_HIGH_REVIEW       1003
HIGH_REVIEW             999
CONSISTENCY_REVIEW       11
Name: count, dtype: int64

Payment anomaly works: 38
Consistency review cases: 11
Payment exceeds sanction: 0

Final score statistics:
count    20016.000000
mean        51.707633
std         24.829550
min          0.251127
50%         48.802596
75%         73.106515
90%         87.856720
95%         92.238709
99%         97.552849
max         99.997502
Name: final_risk_score, dtype: float64


In [28]:
# ==========================================
# FINAL COMBINED RISK ENGINE SUMMARY
# ==========================================

print("=" * 65)
print("MPLADS COMBINED RISK ENGINE — FY 2026-27")
print("=" * 65)

print("\nTotal works:",
      len(risk_df))

print("\nRisk level distribution:")
print(
    risk_df["risk_level"].value_counts()
)

print("\nInvestigation priority:")
print(
    risk_df["investigation_priority"].value_counts()
)

print("\nPayment anomaly works:",
      risk_df["payment_anomaly_flag"].sum())

print("Consistency review cases:",
      risk_df["consistency_review_flag"].sum())

print("Payment exceeds sanction:",
      risk_df["payment_exceeds_sanction_flag"].sum())

print("\nFinal score statistics:")
print(
    risk_df["final_risk_score"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

MPLADS COMBINED RISK ENGINE — FY 2026-27

Total works: 20016

Risk level distribution:
risk_level
LOW          14989
MEDIUM        3025
VERY_HIGH     1003
HIGH           999
Name: count, dtype: int64

Investigation priority:
investigation_priority
NORMAL                18003
VERY_HIGH_REVIEW       1003
HIGH_REVIEW             999
CONSISTENCY_REVIEW       11
Name: count, dtype: int64

Payment anomaly works: 38
Consistency review cases: 11
Payment exceeds sanction: 0

Final score statistics:
count    20016.000000
mean        51.707633
std         24.829550
min          0.251127
50%         48.802596
75%         73.106515
90%         87.856720
95%         92.238709
99%         97.552849
max         99.997502
Name: final_risk_score, dtype: float64


In [29]:
# ==========================================
# EXPORT FINAL RISK ENGINE
# ==========================================

risk_df.to_csv(
    "mplads_final_risk_scores_2026_27.csv",
    index=False
)

print("\nFINAL RISK ENGINE EXPORTED")
print("=" * 50)
print("File: mplads_final_risk_scores_2026_27.csv")
print("Rows:", len(risk_df))
print("Columns:", len(risk_df.columns))


FINAL RISK ENGINE EXPORTED
File: mplads_final_risk_scores_2026_27.csv
Rows: 20016
Columns: 18
